# Embeddings, Recommender Systems, and NN Variants





---


In [ ]:
import os
import glob
import warnings
import random
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

RNG_SEED = 8950
np.random.seed(RNG_SEED)
random.seed(RNG_SEED)
warnings.filterwarnings("ignore")

try:
    import tensorflow as tf
except Exception:
    !pip -q install tensorflow
    import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, regularizers

tf.random.set_seed(RNG_SEED)
print("TensorFlow version:", tf.__version__)


## Shared helpers

These helper functions are provided so the lab can focus on **embeddings, recommendation models, visualization, and interpretation** instead of repetitive plotting boilerplate.


In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def plot_history(history, metric="loss", title=None):
    hist = history.history if hasattr(history, "history") else history
    plt.figure(figsize=(6, 4))
    plt.plot(hist[metric], label=f"train {metric}")
    val_key = "val_" + metric
    if val_key in hist:
        plt.plot(hist[val_key], label=f"valid {metric}")
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.title(title or f"Training history: {metric}")
    plt.legend()
    plt.show()

def standardize_ratings(train_ratings, valid_ratings=None, test_ratings=None):
    mu = float(np.mean(train_ratings))
    sigma = float(np.std(train_ratings) + 1e-8)
    out = [(np.asarray(train_ratings) - mu) / sigma]
    for arr in [valid_ratings, test_ratings]:
        if arr is None:
            out.append(None)
        else:
            out.append((np.asarray(arr) - mu) / sigma)
    return (mu, sigma, *out)

def unstandardize(preds, mu, sigma):
    return np.asarray(preds).reshape(-1) * sigma + mu

def plot_embeddings_2d(Z2, labels=None, title="2D embedding projection", annotate=None):
    plt.figure(figsize=(7, 6))
    if labels is None:
        plt.scatter(Z2[:, 0], Z2[:, 1], s=20, alpha=0.8)
    else:
        labels = np.asarray(labels)
        classes = pd.unique(labels)
        for c in classes:
            mask = labels == c
            plt.scatter(Z2[mask, 0], Z2[mask, 1], s=24, alpha=0.75, label=str(c))
        plt.legend(loc="best", fontsize=8)
    if annotate is not None:
        for (x, y), txt in zip(Z2, annotate):
            plt.text(x, y, str(txt), fontsize=7, alpha=0.8)
    plt.title(title)
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.show()

def get_layer_weights_or_none(model, layer_name):
    try:
        return model.get_layer(layer_name).get_weights()[0]
    except Exception:
        return None


---
## Part A — One-hot vectors, embedding lookup, and similarity

Lecture 10 starts with **symbolic / categorical variables** (user IDs, item IDs, tags, genres, etc.) and explains why we often map them into **dense embedding vectors** instead of keeping a huge sparse one-hot representation.

In this section you will:
- implement one-hot encoding,
- implement a simple embedding lookup,
- compute cosine similarity,
- inspect nearest neighbors in a toy embedding space.


### A1. Implement one-hot encoding and embedding lookup

Fill in:

- `one_hot_encode(ids, vocab_size)`
- `embedding_lookup(ids, E)`

**Hints**
- `ids` is a 1D integer array.
- The result of `one_hot_encode` should have shape `(len(ids), vocab_size)`.
- If `E` is an embedding matrix of shape `(vocab_size, d)`, then `embedding_lookup(ids, E)` should return shape `(len(ids), d)`.


In [ ]:
import numpy as np

def one_hot_encode(ids, vocab_size):
    # TODO
    one_hot = np.zeros((len(ids), vocab_size))
    for i, idx in enumerate(ids):
        one_hot[i, idx] = 1
    return one_hot

def embedding_lookup(ids, E):
    # TODO
    return E[ids]


In [ ]:
toy_tokens = ["user_A", "user_B", "item_X", "item_Y", "item_Z"]
toy_ids = np.array([0, 2, 4, 2, 1])

toy_E = np.array([
    [ 1.00,  0.10,  0.00],
    [ 0.95,  0.05,  0.10],
    [ 0.00,  1.00,  0.10],
    [ 0.05,  0.92,  0.08],
    [ 0.10,  0.15,  0.98],
])

OH = one_hot_encode(toy_ids, vocab_size=len(toy_tokens))
EMB = embedding_lookup(toy_ids, toy_E)

print("One-hot shape:", OH.shape)
print("Embedding lookup shape:", EMB.shape)
display(pd.DataFrame(OH, columns=toy_tokens).head())
display(pd.DataFrame(EMB, columns=["e1", "e2", "e3"]).head())


### A2. Cosine similarity and nearest neighbors

Fill in:

- `cosine_sim(a, b)`
- `topk_similar(query_idx, E, names, k=3)`

**Hints**
- Cosine similarity is
  \[
  \cos(a,b)=
rac{a^	op b}{\|a\|_2\|b\|_2}
  \]
- In `topk_similar`, exclude the query item itself from the returned neighbors.
- Sort from highest similarity to lowest similarity.


In [ ]:
import numpy as np

def cosine_sim(a, b):
    # TODO
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def topk_similar(query_idx, E, names, k=3):
    # TODO
    query_vec = E[query_idx]

    sims = []
    for i in range(len(E)):
        if i != query_idx:
            sim = cosine_sim(query_vec, E[i])
            sims.append((names[i], sim))

    sims.sort(key=lambda x: x[1], reverse=True)

    return sims[:k]


In [ ]:
toy_movie_names = ["Action_1", "Action_2", "Romance_1", "Romance_2", "Comedy_1", "Comedy_2"]
toy_movie_E = np.array([
    [1.00, 0.05, 0.00],
    [0.96, 0.02, 0.06],
    [0.02, 1.00, 0.10],
    [0.08, 0.94, 0.12],
    [0.00, 0.10, 1.00],
    [0.05, 0.12, 0.94],
], dtype=float)

for q_name in ["Action_1", "Romance_1", "Comedy_1"]:
    q_idx = toy_movie_names.index(q_name)
    print(f"Nearest neighbors for {q_name}:")
    print(topk_similar(q_idx, toy_movie_E, toy_movie_names, k=2))
    print()

toy_pca = PCA(n_components=2, random_state=RNG_SEED)
toy_Z2 = toy_pca.fit_transform(toy_movie_E)
plot_embeddings_2d(toy_Z2, labels=[n.split("_")[0] for n in toy_movie_names],
                   title="Toy embedding space (PCA)",
                   annotate=toy_movie_names)


---
## Part B — A tiny recommender warm-up with explicit ratings

We now move from “generic embeddings” to a simple **recommendation model**.

Suppose we observe triples `(user, item, rating)` and want to learn
\[
\hat r_{ui} = \mu + b_u + b_i + p_u^	op q_i
\]
where:
- \(\mu\) is a global mean,
- \(b_u\) is a user bias,
- \(b_i\) is an item bias,
- \(p_u, q_i \in \mathbb{R}^d\) are learned embedding vectors.

In this warm-up, you will implement **one SGD update** and then a small training loop on a toy dataset.


In [ ]:
toy_ratings = pd.DataFrame({
    "user_idx": [0,0,0,1,1,1,2,2,2,3,3,4,4,4,5,5],
    "item_idx": [0,1,3,0,2,4,1,2,5,0,5,1,3,4,2,5],
    "rating":   [5,4,1,4,5,2,1,4,5,5,4,2,5,4,3,5],
})
display(toy_ratings.head())
print("Number of interactions:", len(toy_ratings))


### B1. Implement the matrix-factorization SGD pieces

Fill in:

- `init_mf`
- `predict_mf`
- `sgd_step_mf`
- `fit_mf_sgd`

**Hints**
- Use small Gaussian random initialization for `P` and `Q`.
- One prediction is:
  `mu + bu[u] + bi[i] + np.dot(P[u], Q[i])`
- If `err = pred - r`, the gradient for squared error contributes:
  - `2 * err * Q[i]` to `P[u]`
  - `2 * err * P[u]` to `Q[i]`
- Add L2 regularization to `P[u]`, `Q[i]`, `bu[u]`, and `bi[i]`.


In [ ]:
import numpy as np

def init_mf(n_users, n_items, d=3, seed=RNG_SEED):
    # TODO
    rng = np.random.default_rng(seed)
    mu = 0.0
    P = rng.normal(0, 0.1, size=(n_users, d))
    Q = rng.normal(0, 0.1, size=(n_items, d))
    bu = np.zeros(n_users)
    bi = np.zeros(n_items)
    return mu, P, Q, bu, bi

def predict_mf(u, i, mu, P, Q, bu, bi):
    # TODO
    return mu + bu[u] + bi[i] + np.dot(P[u], Q[i])

def sgd_step_mf(u, i, r, mu, P, Q, bu, bi, lr=0.02, reg=0.01):
    # TODO
    pred = predict_mf(u, i, mu, P, Q, bu, bi)
    err = r - pred

    pu = P[u].copy()
    qi = Q[i].copy()

    bu[u] += lr * (err - reg * bu[u])
    bi[i] += lr * (err - reg * bi[i])
    P[u] += lr * (err * qi - reg * pu)
    Q[i] += lr * (err * pu - reg * qi)

def fit_mf_sgd(df, d=3, lr=0.02, reg=0.01, epochs=200, seed=RNG_SEED):
    # TODO
    n_users = df["user_idx"].nunique()
    n_items = df["item_idx"].nunique()

    mu, P, Q, bu, bi = init_mf(n_users, n_items, d=d, seed=seed)
    mu = df["rating"].mean()

    rng = np.random.default_rng(seed)
    hist = []

    for epoch in range(epochs):
        shuffled_idx = rng.permutation(len(df))
        shuffled_df = df.iloc[shuffled_idx]

        for _, row in shuffled_df.iterrows():
            u = int(row["user_idx"])
            i = int(row["item_idx"])
            r = float(row["rating"])
            sgd_step_mf(u, i, r, mu, P, Q, bu, bi, lr=lr, reg=reg)

        preds = [
            predict_mf(int(u), int(i), mu, P, Q, bu, bi)
            for u, i in df[["user_idx", "item_idx"]].to_numpy()
        ]
        train_rmse = np.sqrt(np.mean((df["rating"].to_numpy() - np.array(preds)) ** 2))
        hist.append(train_rmse)

    return mu, P, Q, bu, bi, hist


In [ ]:
mu_toy, P_toy, Q_toy, bu_toy, bi_toy, hist_toy = fit_mf_sgd(
    toy_ratings, d=3, lr=0.03, reg=0.02, epochs=150
)

plt.figure(figsize=(6, 4))
plt.plot(hist_toy)
plt.title("Toy MF training RMSE")
plt.xlabel("epoch")
plt.ylabel("train RMSE")
plt.show()

toy_preds = [
    predict_mf(int(u), int(i), mu_toy, P_toy, Q_toy, bu_toy, bi_toy)
    for u, i in toy_ratings[["user_idx", "item_idx"]].to_numpy()
]
display(toy_ratings.assign(pred=np.round(toy_preds, 3)))
print("Toy train RMSE:", rmse(toy_ratings["rating"], toy_preds))


### B2. Compare a few tiny MF variants

Try a few embedding dimensions and regularization strengths.  
This is a quick preview of how **model capacity** and **regularization** affect recommender performance.


In [ ]:
mf_results = []
for d in [2, 4, 8]:
    for reg in [0.0, 0.01, 0.05]:
        mu_, P_, Q_, bu_, bi_, hist_ = fit_mf_sgd(
            toy_ratings, d=d, lr=0.03, reg=reg, epochs=120, seed=RNG_SEED
        )
        preds_ = [
            predict_mf(int(u), int(i), mu_, P_, Q_, bu_, bi_)
            for u, i in toy_ratings[["user_idx", "item_idx"]].to_numpy()
        ]
        mf_results.append({
            "d": d,
            "reg": reg,
            "final_train_rmse": rmse(toy_ratings["rating"], preds_),
        })

display(pd.DataFrame(mf_results).sort_values("final_train_rmse"))


---
## Part C — Keras recommender models and NN variants

Lecture 10 also discusses **neural-network-based recommenders**, embeddings inside a network, and **dropout regularization**.

In this section, implement two recommenders with `tf.keras`:

1. **Dot-product recommender**  
   user embedding + item embedding + optional bias terms
2. **MLP recommender**  
   concatenate user/item embeddings, then pass them through dense layers

Later, we will reuse these same builders on the Kaggle dataset.


### C1. Implement the model builders

Fill in:

- `build_dot_model(...)`
- `build_mlp_recommender(...)`

**Hints**
- Use `layers.Embedding` and `layers.Flatten`.
- For the dot model, add user and item bias embeddings with dimension 1.
- For the MLP model, concatenate the flattened user and item embeddings.
- Add `Dropout(dropout)` only if `dropout > 0`.
- Compile with Adam and MSE.


In [ ]:
def build_dot_model(n_users, n_items, embed_dim=16, l2=1e-6, lr=1e-3):
    # TODO
    user_in = keras.Input(shape=(1,), name="user_idx")
    item_in = keras.Input(shape=(1,), name="item_idx")

    user_emb = layers.Embedding(
        input_dim=n_users,
        output_dim=embed_dim,
        embeddings_regularizer=regularizers.l2(l2),
        name="user_embedding",
    )(user_in)
    item_emb = layers.Embedding(
        input_dim=n_items,
        output_dim=embed_dim,
        embeddings_regularizer=regularizers.l2(l2),
        name="item_embedding",
    )(item_in)

    user_vec = layers.Flatten()(user_emb)
    item_vec = layers.Flatten()(item_emb)

    dot = layers.Dot(axes=1)([user_vec, item_vec])

    user_bias = layers.Embedding(
        input_dim=n_users,
        output_dim=1,
        embeddings_regularizer=regularizers.l2(l2),
        name="user_bias",
    )(user_in)
    item_bias = layers.Embedding(
        input_dim=n_items,
        output_dim=1,
        embeddings_regularizer=regularizers.l2(l2),
        name="item_bias",
    )(item_in)

    user_bias = layers.Flatten()(user_bias)
    item_bias = layers.Flatten()(item_bias)

    output = layers.Add()([dot, user_bias, item_bias])

    model = keras.Model(inputs=[user_in, item_in], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
    )
    return model

def build_mlp_recommender(
    n_users,
    n_items,
    embed_dim=32,
    hidden_units=(128, 64),
    dropout=0.2,
    l2=1e-6,
    lr=1e-3,
):
    # TODO
    user_in = keras.Input(shape=(1,), name="user_idx")
    item_in = keras.Input(shape=(1,), name="item_idx")

    user_emb = layers.Embedding(
        input_dim=n_users,
        output_dim=embed_dim,
        embeddings_regularizer=regularizers.l2(l2),
        name="user_embedding",
    )(user_in)
    item_emb = layers.Embedding(
        input_dim=n_items,
        output_dim=embed_dim,
        embeddings_regularizer=regularizers.l2(l2),
        name="item_embedding",
    )(item_in)

    user_vec = layers.Flatten()(user_emb)
    item_vec = layers.Flatten()(item_emb)

    x = layers.Concatenate()([user_vec, item_vec])

    for units in hidden_units:
        x = layers.Dense(
            units,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2),
        )(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)

    output = layers.Dense(1)(x)

    model = keras.Model(inputs=[user_in, item_in], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
    )
    return model


### C2. Sanity check the model builders on synthetic data

We generate a small synthetic dataset from latent factors.  
Your models do **not** need to beat each other here—the goal is to make sure the code trains and produces sensible validation RMSE.


In [ ]:
def make_synthetic_recsys(n_users=250, n_items=400, n_obs=12000, k_true=6, noise=0.30, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    U_true = rng.normal(size=(n_users, k_true))
    V_true = rng.normal(size=(n_items, k_true))
    bu_true = rng.normal(scale=0.2, size=n_users)
    bi_true = rng.normal(scale=0.2, size=n_items)

    user_idx = rng.integers(0, n_users, size=n_obs)
    item_idx = rng.integers(0, n_items, size=n_obs)

    raw = (
        3.5
        + bu_true[user_idx]
        + bi_true[item_idx]
        + np.sum(U_true[user_idx] * V_true[item_idx], axis=1)
        + rng.normal(scale=noise, size=n_obs)
    )
    rating = np.clip(raw, 1.0, 5.0)

    return pd.DataFrame({
        "user_idx": user_idx,
        "item_idx": item_idx,
        "rating": rating.astype(np.float32),
    })

syn = make_synthetic_recsys()
syn_train, syn_valid = train_test_split(syn, test_size=0.2, random_state=RNG_SEED)

mu_syn, sigma_syn, y_syn_train_z, y_syn_valid_z, _ = standardize_ratings(
    syn_train["rating"].to_numpy(),
    syn_valid["rating"].to_numpy(),
    None,
)

callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]

dot_syn = build_dot_model(
    n_users=int(syn["user_idx"].max()) + 1,
    n_items=int(syn["item_idx"].max()) + 1,
    embed_dim=12,
    l2=1e-6,
    lr=1e-3,
)
hist_dot_syn = dot_syn.fit(
    [syn_train["user_idx"], syn_train["item_idx"]],
    y_syn_train_z,
    validation_data=([syn_valid["user_idx"], syn_valid["item_idx"]], y_syn_valid_z),
    epochs=8,
    batch_size=256,
    verbose=0,
    callbacks=callbacks,
)

mlp_syn = build_mlp_recommender(
    n_users=int(syn["user_idx"].max()) + 1,
    n_items=int(syn["item_idx"].max()) + 1,
    embed_dim=16,
    hidden_units=(64, 32),
    dropout=0.20,
    l2=1e-6,
    lr=1e-3,
)
hist_mlp_syn = mlp_syn.fit(
    [syn_train["user_idx"], syn_train["item_idx"]],
    y_syn_train_z,
    validation_data=([syn_valid["user_idx"], syn_valid["item_idx"]], y_syn_valid_z),
    epochs=8,
    batch_size=256,
    verbose=0,
    callbacks=callbacks,
)

plot_history(hist_dot_syn, "loss", title="Synthetic dot-product recommender")
plot_history(hist_mlp_syn, "loss", title="Synthetic MLP recommender")

dot_syn_pred = unstandardize(dot_syn.predict([syn_valid["user_idx"], syn_valid["item_idx"]], verbose=0), mu_syn, sigma_syn)
mlp_syn_pred = unstandardize(mlp_syn.predict([syn_valid["user_idx"], syn_valid["item_idx"]], verbose=0), mu_syn, sigma_syn)

syn_results = pd.DataFrame([
    {"model": "dot", "valid_rmse": rmse(syn_valid["rating"], dot_syn_pred), "valid_mae": mean_absolute_error(syn_valid["rating"], dot_syn_pred)},
    {"model": "mlp_dropout", "valid_rmse": rmse(syn_valid["rating"], mlp_syn_pred), "valid_mae": mean_absolute_error(syn_valid["rating"], mlp_syn_pred)},
])

display(syn_results.sort_values("valid_rmse"))


---
## Part D — Kaggle mini-project: MovieLens 1M recommender with embeddings

In this mini-project you will apply the Lecture 10 ideas to a **real recommender dataset** from Kaggle.

### Mini-project tasks
1. download **MovieLens 1M** from Kaggle,
2. visualize rating patterns and interaction density,
3. preprocess user/item IDs into contiguous integer indices,
4. compare a few recommender variants,
5. visualize learned movie embeddings,
6. retrieve similar movies and generate recommendations for a user.

Recommended workflow:
- start with the provided data-loading helpers,
- complete the TODOs in order,
- save your plots and final results table.


### D0. Kaggle API setup (run once in Colab)

If you are in Google Colab:
1. create a Kaggle API token (`kaggle.json`) from your Kaggle account,
2. upload it when prompted,
3. run the cell below.

If the dataset is already present in `data/`, the cell will simply reuse it.


In [ ]:
!pip -q install kaggle

os.makedirs("data", exist_ok=True)

# If running in Colab and kaggle.json is not configured, upload it.
try:
    from google.colab import files  # type: ignore

    kaggle_dir = os.path.expanduser("~/.kaggle")
    kaggle_json = os.path.join(kaggle_dir, "kaggle.json")
    os.makedirs(kaggle_dir, exist_ok=True)

    if not os.path.exists(kaggle_json):
        print("Upload kaggle.json to enable the Kaggle API.")
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            with open(kaggle_json, "wb") as f:
                f.write(uploaded["kaggle.json"])
            os.chmod(kaggle_json, 0o600)
except Exception:
    pass

need_download = not any(glob.glob("data/**/*ratings*.dat", recursive=True)) and not any(glob.glob("data/**/*ratings*.csv", recursive=True))
if need_download:
    !kaggle datasets download -d odedgolden/movielens-1m-dataset -p data --unzip
else:
    print("MovieLens files already present. Skipping download.")

print("A few files under data/:")
for p in sorted(glob.glob("data/**/*", recursive=True))[:20]:
    print(" ", p)


### D1. Load the MovieLens tables

The standard MovieLens 1M package contains:
- `ratings.dat`
- `movies.dat`
- `users.dat`

We provide the boilerplate loader so the ML work can focus on modeling and analysis.


In [ ]:
def find_first_match(base_dir, candidates):
    matches = []
    for pattern in candidates:
        matches.extend(glob.glob(os.path.join(base_dir, "**", pattern), recursive=True))
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(f"Could not find any of {candidates} under {base_dir}")
    return matches[0]

def load_movielens_1m(base_dir="data"):
    ratings_path = find_first_match(base_dir, ["*ratings*.dat", "*ratings*.csv"])
    movies_path  = find_first_match(base_dir, ["*movies*.dat", "*movies*.csv"])
    users_path   = find_first_match(base_dir, ["*users*.dat", "*users*.csv"])

    if ratings_path.endswith(".dat"):
        ratings = pd.read_csv(
            ratings_path,
            sep="::",
            engine="python",
            header=None,
            names=["userId", "movieId", "rating", "timestamp"],
            encoding="latin-1",
        )
    else:
        ratings = pd.read_csv(ratings_path)
        ratings.columns = [c.strip() for c in ratings.columns]
        ratings = ratings.rename(columns={
            ratings.columns[0]: "userId",
            ratings.columns[1]: "movieId",
            ratings.columns[2]: "rating",
        })

    if movies_path.endswith(".dat"):
        movies = pd.read_csv(
            movies_path,
            sep="::",
            engine="python",
            header=None,
            names=["movieId", "title", "genres"],
            encoding="latin-1",
        )
    else:
        movies = pd.read_csv(movies_path)
        movies.columns = [c.strip() for c in movies.columns]

    if users_path.endswith(".dat"):
        users = pd.read_csv(
            users_path,
            sep="::",
            engine="python",
            header=None,
            names=["userId", "gender", "age", "occupation", "zip"],
            encoding="latin-1",
        )
    else:
        users = pd.read_csv(users_path)
        users.columns = [c.strip() for c in users.columns]

    return ratings, movies, users

ratings_raw, movies_raw, users_raw = load_movielens_1m("data")

print("ratings:", ratings_raw.shape)
print("movies :", movies_raw.shape)
print("users  :", users_raw.shape)

display(ratings_raw.head())
display(movies_raw.head())
display(users_raw.head())


### D2. Explore the raw data

Complete the following EDA preparation cell.

Compute:
- `ratings_per_user`
- `ratings_per_movie`
- `genre_counts`

**Hints**
- `ratings_per_user` should be a Series indexed by `userId`.
- `ratings_per_movie` should be a Series indexed by `movieId`.
- `genre_counts` can be built by splitting the `genres` strings on `"|"`.


In [ ]:
# TODO
ratings_per_user = ratings_raw.groupby("userId").size()
ratings_per_movie = ratings_raw.groupby("movieId").size()

if "genres" in ratings_raw.columns:
    genre_source = ratings_raw
elif "movies" in globals() and "genres" in movies.columns:
    genre_source = movies
elif "movies_raw" in globals() and "genres" in movies_raw.columns:
    genre_source = movies_raw
elif "movies_df" in globals() and "genres" in movies_df.columns:
    genre_source = movies_df
else:
    genre_source = None

if genre_source is not None:
    genre_counts = (
        genre_source["genres"]
        .dropna()
        .str.split("|")
        .explode()
        .value_counts()
    )
else:
    genre_counts = pd.Series([0], index=["Unknown"])

# ratings_per_user = ...
# ratings_per_movie = ...
# genre_counts = ...

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(ratings_raw["rating"], bins=np.arange(0.5, 5.6, 0.5), edgecolor="black")
axes[0].set_title("Ratings distribution")
axes[0].set_xlabel("rating")
axes[0].set_ylabel("count")

axes[1].hist(ratings_per_user.values, bins=40, edgecolor="black")
axes[1].set_title("Ratings per user")
axes[1].set_xlabel("# ratings")
axes[1].set_ylabel("count")
axes[1].set_yscale("log")

genre_counts.head(12).sort_values().plot(kind="barh", ax=axes[2])
axes[2].set_title("Top genres")
axes[2].set_xlabel("count")

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(ratings_per_movie.values, bins=40, edgecolor="black")
plt.yscale("log")
plt.xlabel("# ratings for a movie")
plt.ylabel("count")
plt.title("Movie popularity distribution")
plt.show()


### D3. Prepare a modeling table

Build a cleaned interaction table named `ratings_df` with the following steps:

1. keep only users with at least `min_user_ratings` interactions,
2. keep only movies with at least `min_movie_ratings` interactions,
3. optionally sample up to `sample_n` rows for faster Colab training,
4. factorize `userId` into `user_idx`,
5. factorize `movieId` into `movie_idx`,
6. merge in `title` and `genres`.

**Output requirement**
Your final DataFrame should contain at least these columns:
`["userId", "movieId", "rating", "user_idx", "movie_idx", "title", "genres"]`


In [ ]:
def prepare_movielens_frame(
    ratings,
    movies,
    min_user_ratings=20,
    min_movie_ratings=20,
    sample_n=250_000,
    seed=RNG_SEED,
):
    # TODO
    ratings = ratings.copy()
    movies = movies.copy()

    user_counts = ratings["userId"].value_counts()
    movie_counts = ratings["movieId"].value_counts()

    valid_users = user_counts[user_counts >= min_user_ratings].index
    valid_movies = movie_counts[movie_counts >= min_movie_ratings].index

    ratings = ratings[
        ratings["userId"].isin(valid_users) &
        ratings["movieId"].isin(valid_movies)
    ].copy()

    if sample_n is not None and len(ratings) > sample_n:
        ratings = ratings.sample(n=sample_n, random_state=seed).copy()

    ratings = ratings.reset_index(drop=True)

    movie_cols = ["movieId"]
    if "title" in movies.columns:
        movie_cols.append("title")
    if "genres" in movies.columns:
        movie_cols.append("genres")

    ratings = ratings.merge(
        movies[movie_cols].drop_duplicates("movieId"),
        on="movieId",
        how="left"
    )

    user_codes, user_uniques = pd.factorize(ratings["userId"], sort=True)
    movie_codes, movie_uniques = pd.factorize(ratings["movieId"], sort=True)

    ratings["user_idx"] = user_codes.astype(int)
    ratings["movie_idx"] = movie_codes.astype(int)

    cols = ["userId", "movieId", "rating", "user_idx", "movie_idx"]
    if "timestamp" in ratings.columns:
        cols.append("timestamp")
    if "title" in ratings.columns:
        cols.append("title")
    if "genres" in ratings.columns:
        cols.append("genres")

    ratings = ratings[cols].copy()

    return ratings

ratings_df = prepare_movielens_frame(
    ratings_raw, movies_raw,
    min_user_ratings=20,
    min_movie_ratings=20,
    sample_n=250_000,
    seed=RNG_SEED,
)


In [ ]:
print("Prepared interactions:", ratings_df.shape)
print("Unique users:", ratings_df["user_idx"].nunique())
print("Unique movies:", ratings_df["movie_idx"].nunique())
display(ratings_df.head())

assert ratings_df["user_idx"].min() == 0
assert ratings_df["movie_idx"].min() == 0
assert ratings_df["user_idx"].max() + 1 == ratings_df["user_idx"].nunique()
assert ratings_df["movie_idx"].max() + 1 == ratings_df["movie_idx"].nunique()


### D4. Train / validation / test split

We do a simple random split by observed ratings:
- 70% train
- 15% validation
- 15% test

Later, when you interpret results, remember that this is still an **explicit-feedback rating prediction** setup—not a full temporal production recommender evaluation.


In [ ]:
train_df, temp_df = train_test_split(ratings_df, test_size=0.30, random_state=RNG_SEED)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=RNG_SEED)

print("train:", train_df.shape, "valid:", valid_df.shape, "test:", test_df.shape)


### D5. Baselines

First compute two simple baselines:

1. **global mean** baseline  
   always predict the mean training rating
2. **movie mean** baseline  
   predict the average training rating for each movie, and fall back to the global mean for unseen movies

Fill in `movie_mean_predict`.


In [ ]:
global_mean = float(train_df["rating"].mean())

def global_mean_predict(n, value):
    return np.full(n, fill_value=value, dtype=float)


In [ ]:
def movie_mean_predict(train_df, eval_df, global_mean):
    # TODO
    movie_means = train_df.groupby("movie_idx")["rating"].mean()
    preds = eval_df["movie_idx"].map(movie_means).fillna(global_mean)
    return preds.to_numpy()


In [ ]:
base_results = []

pred_global_valid = global_mean_predict(len(valid_df), global_mean)
pred_movie_valid = movie_mean_predict(train_df, valid_df, global_mean)

base_results.append({
    "model": "global_mean",
    "valid_rmse": rmse(valid_df["rating"], pred_global_valid),
    "valid_mae": mean_absolute_error(valid_df["rating"], pred_global_valid),
})
base_results.append({
    "model": "movie_mean",
    "valid_rmse": rmse(valid_df["rating"], pred_movie_valid),
    "valid_mae": mean_absolute_error(valid_df["rating"], pred_movie_valid),
})

display(pd.DataFrame(base_results).sort_values("valid_rmse"))


### D6. Train embedding-based recommenders

Reuse your model builders from Part C.

Compare at least these models:
- `dot_small` (e.g. embedding dim 16)
- `dot_large` (e.g. embedding dim 48)
- `mlp_dropout` (MLP with dropout)
- one extra variant of your choice

**Tip**
Standardizing the training targets often makes neural training more stable.


In [ ]:
n_users = int(ratings_df["user_idx"].max()) + 1
n_movies = int(ratings_df["movie_idx"].max()) + 1

mu_rating, sigma_rating, y_train_z, y_valid_z, y_test_z = standardize_ratings(
    train_df["rating"].to_numpy(dtype=float),
    valid_df["rating"].to_numpy(dtype=float),
    test_df["rating"].to_numpy(dtype=float),
)

X_train = [train_df["user_idx"].to_numpy(dtype=np.int32), train_df["movie_idx"].to_numpy(dtype=np.int32)]
X_valid = [valid_df["user_idx"].to_numpy(dtype=np.int32), valid_df["movie_idx"].to_numpy(dtype=np.int32)]
X_test  = [test_df["user_idx"].to_numpy(dtype=np.int32),  test_df["movie_idx"].to_numpy(dtype=np.int32)]

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)


Fill in the training code for your required model variants below.  
Keep the number of epochs modest so the notebook remains Colab-friendly.


In [ ]:
# TODO
# Build and train at least:
#   - dot_small
#   - dot_large
#   - mlp_dropout
#   - one extra variant of your choice
#
# Expected outputs:
#   fitted_models  : dict[str, keras.Model]
#   histories      : dict[str, History]
#   model_results  : list[dict]
#   results_df     : DataFrame with baseline + neural results

fitted_models = {}
histories = {}
model_results = []

model_builders = {
    "dot_small": lambda: build_dot_model(
        n_users=n_users,
        n_items=n_movies,
        embed_dim=16,
        l2=1e-6,
        lr=1e-3,
    ),
    "dot_large": lambda: build_dot_model(
        n_users=n_users,
        n_items=n_movies,
        embed_dim=48,
        l2=1e-6,
        lr=1e-3,
    ),
    "mlp_dropout": lambda: build_mlp_recommender(
        n_users=n_users,
        n_items=n_movies,
        embed_dim=32,
        hidden_units=(128, 64),
        dropout=0.2,
        l2=1e-6,
        lr=1e-3,
    ),
    "mlp_small": lambda: build_mlp_recommender(
        n_users=n_users,
        n_items=n_movies,
        embed_dim=16,
        hidden_units=(64, 32),
        dropout=0.0,
        l2=1e-6,
        lr=1e-3,
    ),
}

for model_name, build_fn in model_builders.items():
    model = build_fn()

    history = model.fit(
        X_train,
        y_train_z,
        validation_data=(X_valid, y_valid_z),
        epochs=10,
        batch_size=512,
        callbacks=[early_stop],
        verbose=0,
    )

    pred_valid_z = model.predict(X_valid, verbose=0).reshape(-1)
    pred_valid = unstandardize(pred_valid_z, mu_rating, sigma_rating)

    fitted_models[model_name] = model
    histories[model_name] = history

    model_results.append({
        "model": model_name,
        "valid_rmse": rmse(valid_df["rating"], pred_valid),
        "valid_mae": mean_absolute_error(valid_df["rating"], pred_valid),
    })

results_df = pd.DataFrame(base_results + model_results).sort_values("valid_rmse").reset_index(drop=True)


In [ ]:
try:
    display(results_df)
except NameError:
    print("results_df is not defined yet. Finish the training cell above.")


### D7. Inspect training curves

Plot the loss curves for at least two neural recommender variants and comment on:
- convergence speed,
- whether dropout helps,
- whether larger embedding dimension helps.


In [ ]:
try:
    for name in list(histories.keys())[:3]:
        plot_history(histories[name], "loss", title=f"{name} training curve")
except NameError:
    print("histories is not defined yet. Finish the training cell above.")


### D8. Visualize learned movie embeddings

Use the **best neural recommender** (lowest validation RMSE) and extract its movie embedding matrix.

Fill in:
- `get_movie_embedding_matrix(model)`
- `dominant_genre(genres)`


In [ ]:
def get_movie_embedding_matrix(model):
    # TODO
    for layer in model.layers:
        if "item_embedding" in layer.name or "movie_embedding" in layer.name:
            return layer.get_weights()[0]
    raise ValueError("Could not find movie/item embedding layer in the model")

def dominant_genre(genres):
    # TODO
    if pd.isna(genres):
        return "Unknown"
    return str(genres).split("|")[0]

In [ ]:
try:
    neural_only = results_df[~results_df["model"].isin(["global_mean", "movie_mean"])].copy()
    best_model_name = neural_only.sort_values("valid_rmse").iloc[0]["model"]
    best_model = fitted_models[best_model_name]
    print("Best neural model:", best_model_name)

    movie_emb = get_movie_embedding_matrix(best_model)

    movie_pop = train_df.groupby("movie_idx").size().sort_values(ascending=False)
    top_movie_idx = movie_pop.head(250).index.to_numpy()

    movie_meta = (
        ratings_df[["movie_idx", "title", "genres"]]
        .drop_duplicates("movie_idx")
        .set_index("movie_idx")
        .loc[top_movie_idx]
        .reset_index()
    )

    Z = movie_emb[top_movie_idx]
    Z2_pca = PCA(n_components=2, random_state=RNG_SEED).fit_transform(Z)

    labels = movie_meta["genres"].map(dominant_genre).to_numpy()
    plot_embeddings_2d(
        Z2_pca,
        labels=labels,
        title=f"Movie embeddings (PCA) — {best_model_name}",
        annotate=None,
    )

    # Optional t-SNE on a smaller subset
    tsne_subset = min(150, len(top_movie_idx))
    Z_tsne = TSNE(
        n_components=2,
        perplexity=25,
        init="random",
        learning_rate="auto",
        random_state=RNG_SEED,
    ).fit_transform(Z[:tsne_subset])

    plot_embeddings_2d(
        Z_tsne,
        labels=labels[:tsne_subset],
        title=f"Movie embeddings (t-SNE subset) — {best_model_name}",
        annotate=None,
    )
except NameError:
    print("Complete the model-training cells first.")


### D9. Similar movies from embedding space

Fill in `find_similar_movies` to retrieve nearest neighbors for a movie title using cosine similarity in the learned embedding space.

**Hints**
- Use case-insensitive matching to find a movie whose title contains the query text.
- Exclude the anchor movie itself.
- Return a small DataFrame with title, genres, and similarity score.


In [ ]:
def find_similar_movies(title_query, movie_emb, ratings_df, k=10):
    # TODO
    movies_meta = (
        ratings_df[["movie_idx", "title", "genres"]]
        .drop_duplicates(subset=["movie_idx"])
        .reset_index(drop=True)
    )

    matches = movies_meta[movies_meta["title"].str.contains(title_query, case=False, na=False)]

    if len(matches) == 0:
        raise ValueError(f"No movie title found matching: {title_query}")

    query_row = matches.iloc[0]
    query_idx = int(query_row["movie_idx"])
    query_vec = movie_emb[query_idx]

    norms = np.linalg.norm(movie_emb, axis=1)
    query_norm = np.linalg.norm(query_vec)

    sims = movie_emb @ query_vec / (norms * query_norm + 1e-12)

    sim_df = movies_meta.copy()
    sim_df["similarity"] = sim_df["movie_idx"].map(lambda idx: sims[int(idx)])
    sim_df["dominant_genre"] = sim_df["genres"].apply(dominant_genre)

    sim_df = sim_df[sim_df["movie_idx"] != query_idx]
    sim_df = sim_df.sort_values("similarity", ascending=False).head(k).reset_index(drop=True)

    return sim_df[["movie_idx", "title", "genres", "dominant_genre", "similarity"]]


In [ ]:
try:
    anchor_title, neighbors_df = find_similar_movies("Toy Story", movie_emb, ratings_df, k=8)
    print("Anchor movie:", anchor_title)
    display(neighbors_df)
except Exception as e:
    print("Similar-movie lookup could not run yet:", e)


### D10. Top-N recommendations for one user

Fill in `recommend_for_user`.

Given a raw `userId`:
1. find the corresponding `user_idx`,
2. collect the movies already seen by that user in the training data,
3. score all unseen movies with the best neural model,
4. return the top recommendations.

**Hint**
- The neural models were trained on **standardized ratings**, so unstandardize predictions before sorting.


In [ ]:
def recommend_for_user(user_id, model, train_df, ratings_df, mu_rating, sigma_rating, k=10):
    # TODO
    all_movies = (
        ratings_df[["movie_idx", "movieId", "title", "genres"]]
        .drop_duplicates(subset=["movie_idx"])
        .reset_index(drop=True)
    )

    seen_movies = set(train_df.loc[train_df["user_idx"] == user_id, "movie_idx"].tolist())

    candidates = all_movies[~all_movies["movie_idx"].isin(seen_movies)].copy()

    if len(candidates) == 0:
        return candidates

    user_array = np.full(len(candidates), user_id, dtype=np.int32)
    movie_array = candidates["movie_idx"].to_numpy(dtype=np.int32)

    preds_z = model.predict(
        {"user_idx": user_array, "item_idx": movie_array},
        verbose=0
    ).reshape(-1)

    preds = preds_z * sigma_rating + mu_rating

    candidates["pred_rating"] = preds
    candidates["dominant_genre"] = candidates["genres"].apply(dominant_genre)

    candidates = candidates.sort_values("pred_rating", ascending=False).head(k).reset_index(drop=True)

    return candidates[["movie_idx", "movieId", "title", "genres", "dominant_genre", "pred_rating"]]

In [ ]:
try:
    active_user = int(train_df["userId"].value_counts().index[0])
    print("Example userId:", active_user)
    recs_df = recommend_for_user(active_user, best_model, train_df, ratings_df, mu_rating, sigma_rating, k=10)
    display(recs_df)
except Exception as e:
    print("Recommendation demo could not run yet:", e)


---
## Reflection questions (write short answers in Markdown)

1. Which recommender variant achieved the best **validation RMSE**, and why do you think it worked best?

The MLP with dropout worked best. It captures more complex user–movie interactions than simple dot product, and dropout helped reduce overfitting.

2. Did increasing embedding dimension always help?

No. After a point, increasing embedding size didn’t improve much and sometimes made it worse due to overfitting.

3. Did the movie neighbors found in embedding space look semantically reasonable?

Yes, mostly. Similar genres and related movies were grouped together, which shows the embeddings learned meaningful patterns.

4. Name one possible **bias / fairness / popularity** concern in this recommendation setup.

Popular movies get recommended more because they have more data, so less popular or niche movies may be ignored.

5. If you had more time, what would you improve next?  
   (Examples: better split, temporal evaluation, side metadata, ranking loss, negative sampling, triplet loss, fairness audit.)

   I would add temporal evaluation and use ranking-based loss instead of just RMSE to make recommendations more realistic.
